In [20]:
from safetensors.torch import load_file as safeload
import torch
import torch.nn as nn
from accelerate.state import PartialState
_ = PartialState()
from data_utils.vocabs import TaxaVocabulary
import os
from safetensors.torch import save_file
import shutil



In [5]:
import sys
sys.path.append("/home/kchen/microbiome/gut_microbiome_GPT")

In [6]:
import json
from model.model import hgmGPT


def load_trained_model(model_config_path, checkpoint_path):
    """
    Load trained model from checkpoint for inference.
    
    :param cfg: Configuration object.
    :param model_config: Complete model configuration.
    :param accelerator: Accelerator instance.
    :return: Loaded model.
    """
    # load config from cfg.paths.model_config_path if exists, otherwise build from current cfg
    print(f"Loading model configuration from {model_config_path}...")
    with open(model_config_path, 'r') as f:
        model_config = json.load(f)

    model = hgmGPT(**model_config)
    
    print(f"Loading model checkpoint from {checkpoint_path}...")
    if checkpoint_path.endswith('.pt') or checkpoint_path.endswith('.pth'):
        loaded_state = torch.load(checkpoint_path, map_location='cpu')['model']
    elif checkpoint_path.endswith('.safetensors'):
        from safetensors.torch import load_file
        loaded_state = load_file(checkpoint_path)
        
    model.load_state_dict(loaded_state)
    model.eval()
    
    print("Model loaded and set to evaluation mode.")
    return model

graph_data

1. add to taxa_vocab at end
2. in encoder, add additional embeddings to node_embs and add to num_taxa accordingly
3. graph_data also needs to be updated


In [10]:
import torch

def add_leaf_node(data, parent_idx, new_node_name, new_x_feat):
    # 1. The new index is the current number of nodes
    new_idx = data.num_nodes 
    
    # 2. Append to Node Features (x)
    data.x = torch.cat([data.x, new_x_feat.view(1, -1)], dim=0)
    
    # 3. Add the Edge (Parent -> Leaf)
    # Most taxonomic trees are directed (Parent to Child) 
    # or undirected (both directions).
    new_edge = torch.tensor([[parent_idx], [new_idx]], dtype=torch.long)
    data.edge_index = torch.cat([data.edge_index, new_edge], dim=1)
    
    # 4. Update the Artificial Keys
    data.names.append(new_node_name)
    data.name_to_idx[new_node_name] = new_idx
    data.num_nodes += 1
    
    # 5. Update parent_children dict (if it exists)
    if parent_idx in data.parent_children:
        data.parent_children[parent_idx].append(new_idx)
    else:
        data.parent_children[parent_idx] = [new_idx]
        
    return data

In [11]:
import torch
import copy

def add_leaf_to_graph_object(data, parent_name, leaf_name, full_name, taxa_vocab, model, distance=1.0):
    # 1. Define the new index 'a'
    # Adding to the end is safest to avoid shifting existing tensors
    data = data.clone()  # Clone to avoid in-place modifications if not desired
    taxa_vocab = copy.deepcopy(taxa_vocab)  # Clone to avoid in-place modifications if not desired
    model = copy.deepcopy(model)  # Clone to avoid in-place modifications if not desired
    new_node_idx = model.taxa_encoder.node_embs.num_embeddings
    new_vocab_idx = len(taxa_vocab)  # This is the next available index for the new taxa in the vocab
    
    # assert full_name ends with leaf_name
    assert full_name.endswith(leaf_name), "Full name should end with leaf name"
    
    # 1. Search for a parent that ends with parent_name
    matching_parents = [name for name in data.name_to_idx if name.endswith(parent_name)]

    if not matching_parents:
        raise ValueError(f"Parent {parent_name} not found in graph.")

    # 2. Pick the match (ideally the specific one)
    parent_full_name = matching_parents[0]

    # 3. Print warning if it's not an exact match
    if parent_full_name != parent_name:
        print(f"Warning: Exact parent '{parent_name}' not found. Using match: '{parent_full_name}'")

    parent_idx = data.name_to_idx[parent_full_name]

    # 2. Update Natural Hierarchy (Metadata)
    data.names.append(leaf_name)
    data.name_to_idx[leaf_name] = new_node_idx
    
    # Child -> Parent mapping
    data.child_parents[new_node_idx] = parent_idx
    
    # Parent -> List of Children mapping
    if parent_idx not in data.parent_children:
        data.parent_children[parent_idx] = []
    data.parent_children[parent_idx].append(new_node_idx)
    
    # Distance mapping (stored as a dictionary key tuple)
    data.parent_child_distances[(parent_idx, new_node_idx)] = distance

    # 3. Update Artificial/PyG Mapping
    print(f"new_vocab_idx: {new_vocab_idx}, new node_idx: {new_node_idx}")
    # Create a 1D tensor for the new index
    new_element = torch.tensor([new_node_idx], dtype=torch.long)
    # # 1. Calculate how many elements to add (0 if already long enough)
    # padding_size = max(0, new_vocab_idx - len(data.vocabindex_to_nodeindex) + 1)

    # # 2. Pad with -1 and assign the new node index
    # data.vocabindex_to_nodeindex = torch.nn.functional.pad(data.vocabindex_to_nodeindex, (0, padding_size), value=-1)
    data.vocabindex_to_nodeindex = torch.cat([data.vocabindex_to_nodeindex, new_element])
    assert data.vocabindex_to_nodeindex[new_vocab_idx] == new_node_idx, f"vocabindex_to_nodeindex should map new_vocab_idx {new_vocab_idx} to new_node_idx {new_node_idx}"
    
    # also expand taxa_vocab
    taxa_vocab.add_taxa(full_name, new_vocab_idx)
    assert taxa_vocab.token_to_id[full_name] == new_vocab_idx, "New full name should be added to taxa_vocab with correct index"
    
    # also expand model's taxa encoder
    num_nodes, emb_dim = model.taxa_encoder.node_embs.weight.shape
    new_layer = nn.Embedding(num_nodes + 1, emb_dim).to(model.taxa_encoder.node_embs.weight.device)
    assert new_layer.weight.shape[0] == new_node_idx + 1, f"new layer shape {new_layer.weight.shape} should accommodate new_vocab_idx {new_vocab_idx}"
    
    with torch.no_grad():
        new_layer.weight[:num_nodes] = model.taxa_encoder.node_embs.weight
        new_layer.weight[num_nodes:] = model.taxa_encoder.node_embs.weight.mean(dim=0)
    model.taxa_encoder.node_embs = new_layer
    model.taxa_encoder.num_taxa += 1

    # Add the edge: [2, E+1]
    new_edge = torch.tensor([[parent_idx], [new_node_idx]], dtype=torch.long)
    data.edge_index = torch.cat([data.edge_index, new_edge], dim=1)
    
    data.edge_attr = torch.cat([data.edge_attr, torch.tensor([[1]], dtype=data.edge_attr.dtype)], dim=0)

    # 5. Increment node count
    data.num_nodes += 1
    
    return data, taxa_vocab, model

In [21]:

seed_taxa = [
    "Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Adlercreutzia",
    "Bacteria.Bacillota.Bacilli.Lactobacillales.Lactobacillaceae.Limosilactobacillus",
    "Archaea.Methanobacteriota.Methanobacteria.Methanobacteriales.Methanobacteriaceae.Methanobrevibacter",
    "Bacteria.Bacillota.Bacilli.Erysipelotrichales.Erysipelatoclostridiaceae.Catenibacterium",
    "Bacteria.Bacillota.Clostridia.Peptostreptococcales-Tissierellales.Anaerovoracaceae.Mogibacterium",
]
original_dir = "/home/kchen/microbiome/gut_microbiome_GPT/outputs/pretrain_censored"
new_output_dir = "/home/kchen/microbiome/gut_microbiome_GPT/outputs/pretrain_censored_with_seed_taxa"


taxa_graph = torch.load(
    os.path.join(original_dir, "taxonomic_graph.pt"),
    weights_only=False
)
model = load_trained_model(
    model_config_path=os.path.join(original_dir, "model_config.json"),
    checkpoint_path=os.path.join(original_dir, "best_model/best_model/model.safetensors")
)
taxa_vocab = TaxaVocabulary.load(
    os.path.join(original_dir, "taxa_vocab.pkl")
)

curr_data = taxa_graph.clone()  # Clone to avoid in-place modifications if not desired
curr_vocab = copy.deepcopy(taxa_vocab)  # Clone to avoid in-place modifications if not desired
curr_model = copy.deepcopy(model) 
with open("/home/kchen/microbiome/gut_microbiome_GPT/outputs/pretrain_censored/model_config.json", 'r') as f:
    model_config = json.load(f)
model_config = {
    **model_config,  # assuming this exists from earlier load
    "num_taxa": len(curr_vocab)
}

for full_name in seed_taxa:
    # Extract parent and leaf (assuming '.' delimiter)
    parts = full_name.split('.')
    parent_name = parts[-2]
    leaf_name = parts[-1]
    
    # Update objects in-place for the next iteration
    curr_data, curr_vocab, curr_model = add_leaf_to_graph_object(
        data=curr_data,
        taxa_vocab=curr_vocab,
        model=curr_model,
        parent_name=parent_name,
        leaf_name=leaf_name,
        full_name=full_name,
        distance=1.0
    )

print(f"Final vocab size: {len(curr_vocab)}")
print(f"Final graph nodes: {curr_data.num_nodes}")


os.makedirs(new_output_dir, exist_ok=True)

# 1. Save graph
torch.save(curr_data, os.path.join(new_output_dir, "taxonomic_graph.pt"))

# 2. Save vocab
curr_vocab.save(os.path.join(new_output_dir, "taxa_vocab.pkl"))

# 3. Save model weights (safetensors format)
save_file(curr_model.state_dict(), os.path.join(new_output_dir, "model.safetensors"))

# 5. Copy batch_vocab.pkl if it exists
batch_vocab_path = os.path.join(original_dir, "batch_vocab.pkl")
if os.path.exists(batch_vocab_path):
    shutil.copy2(batch_vocab_path,
                 os.path.join(new_output_dir, "batch_vocab.pkl"))
    print("Copied batch_vocab.pkl")
else:
    print("No batch_vocab.pkl found — skipping.")

Loading model configuration from /home/kchen/microbiome/gut_microbiome_GPT/outputs/pretrain_censored/model_config.json...
trainers - INFO - Initialized hgmGPT model with 878209 trainable parameters
trainers - INFO - Model arguments:
trainers - INFO - 	 d_model: 128
trainers - INFO - 	 use_batch_labels: True
trainers - INFO - 	 num_batch_labels: 40
trainers - INFO - 	 abundance_emb_style: continuous
trainers - INFO - 	 nhead: 8
trainers - INFO - 	 tasks: ['masking']
trainers - INFO - 	 sample_emb_style: cls
trainers - INFO - 	 dropout: 0.1
trainers - INFO - 	 model_distribution: None
trainers - INFO - 	 use_gnn: True
trainers - INFO - 	 num_taxa (vocab size): 1509
trainers - INFO - hgmGPT(
  (taxa_encoder): TaxaGraphEncoder(
    (node_embs): Embedding(1510, 128)
    (conv_model): GAT(128, 128, num_layers=2)
  )
  (value_encoder): ContinuousValueEncoder(
    (dropout): Dropout(p=0.1, inplace=False)
    (linear1): Linear(in_features=1, out_features=128, bias=True)
    (activation): ReLU()